In [1]:
import instructor
from enum import Enum
from openai import AzureOpenAI
from functools import partial
# from Entity_response_model import Demographic, Refractory_Relapse, Blast, Diagnosis, test
from Entity_IC_test import Test,Demographcics,Inhibitors
from Entity_IC import Entity as Entity_IC
from Entity_EC import Entity as Entity_EC
import pandas as pd
from datetime import datetime
from typing import TypeVar, Type, Optional, List
from pydantic import BaseModel, Field
import os
import re
from itertools import islice


In [2]:
# Define your Azure OpenAI configuration
model = "gpt4o_cursor"
engine = "gpt4o_cursor"
endpoint = "https://gpt4o-xj.openai.azure.com/"

api_version = "2024-02-15-preview"
api_type = "azure"

# key_name = "AZURE_KEY"
api_key = 'YOUR-API-KEY'

# Create the Azure OpenAI client
client = AzureOpenAI(
    api_version=api_version,
    api_key=api_key,
    azure_endpoint=endpoint,
    azure_deployment=engine
)

# Patch the client with instructor
client = instructor.patch(client)

# Optionally, create a partial function with default parameters
in_client = partial(
    client.chat.completions.create,
    model=model,
    max_retries=2  # Retries in case of API issues
)


In [3]:
input_dir = '../input'
df = pd.DataFrame(columns=['File Name', 'Inclusion Criteria', 'Exclusion Criteria'])
abstract_list = [] 

for file_name in os.listdir(input_dir):
    if file_name.endswith('.txt'):
        file_path = os.path.join(input_dir, file_name)
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            ### Split the notes to inclusion and exclusion criteria seperately
            inclusion_criteria = content.split('Exclusion Criteria:')[0].split('Inclusion Criteria:')[-1].strip()
            exclusion_criteria = content.split('Exclusion Criteria:')[-1].strip()
            new_row = pd.DataFrame({'File Name': [file_name], 'Inclusion Criteria': [inclusion_criteria], 'Exclusion Criteria': [exclusion_criteria]})
            df = pd.concat([df, new_row], ignore_index=True)

abstract_list = df[['File Name', 'Inclusion Criteria', 'Exclusion Criteria']].values.tolist()
print("Abstracts have been successfully read into a list.")

Abstracts have been successfully read into a list.


In [4]:
T = TypeVar("T", bound=BaseModel)

class InfectionClassifier:
    def __init__(self, response_model: Type[T], model_name: str, client):
        self.response_model = response_model
        self.model_name = model_name
        self.client = client

    def ask_ai(self, criteria: str, type: str) -> T:
        return self.client(
            response_model=self.response_model,
            temperature=0,
            messages=[
                {
                    "role": "system",
                    "content": "You will help extract entities from clinical protocol documents.",
                },
                {
                    "role": "user",
                    "content": f"Extract information from the clinical trial eligibility document ({type} criteria).",
                },
                {
                    "role": "user",
                    "content": f"Here is the {type} Criteria section:\n{criteria}",
                },
                {
                    "role": "user",
                    "content": f'''Please follow the following guidelines to answer the questions:
                                * Ensure you distinguish between 'CYP3A4' and 'CYP3A'. They are distinct, and an instance of 'CYP3A' should not be interpreted as 'CYP3A4'.
                                * Given content 'Refractory leukemia, defined as persistent leukemia after at least two courses of induction chemotherapy', 
                                it does not indicate occurrences of refractory disease but indicates the number of cycles of therapy required for refractory disease.
                                * Given content 'evidence of leukemia with ≥ 5% blasts in the blood', it dose not indicate blood absolute blast count but indicates blood blast percentage.
                                * Given content 'At least 24 hours must have elapsed since the completion of low-dose or non- myelosuppressive therapy', 
                                it indicates prior exposure to non-myelosuppressive anti-cancer agents.
                                * Given content 'At least 14 days must have elapsed since the completion of myelosuppressive therapy', it indicates prior exposure to myelosuppressive anti-cancer agents.
                                ''',
                },
            ],
        )

    def output(self, criteria: str, type: str):
        try:
            response = self.ask_ai(criteria, type)
            entity_dict = response.dict()

            for key, value in entity_dict.items():
                if isinstance(value, list):
                    if value and isinstance(value[0], Enum):
                        # Extract the string representation and join list elements if necessary
                        entity_dict[key] = ", ".join(v.value for v in value)
                    elif value:  # For regular lists
                        entity_dict[key] = ", ".join(str(v) for v in value)
                    else:
                        entity_dict[key] = ""  # Empty list -> None
                elif isinstance(value, Enum):
                    entity_dict[key] = value.value  # For single Enum fields

            entity_value_data = {
                "Entity": list(entity_dict.keys()),
                "Value": list(entity_dict.values()),
            }
            df = pd.DataFrame(entity_value_data)
            return df

        except Exception as e:
            print(f"Error in output method: {str(e)}")
            # Return an empty DataFrame if an error occurs
            return pd.DataFrame(columns=["Entity", "Value"])


In [5]:
# Load the uploaded Excel file to inspect its content
file_path = '../GEARBOx Ontology for LLM_In use.xlsx'
excel_data = pd.ExcelFile(file_path)

# Display the sheet names to understand the structure of the file
excel_data.sheet_names

['GEARBOx v5',
 'GEARBOx v4',
 'GEARBOx v3',
 'Questions & Comments',
 'GEARBOx v2-Kai',
 'GEARBOx v2',
 'GEARBOx v1',
 'Sheet1',
 'Sheet2',
 'Sheet3',
 'xx-GEARBOx',
 'xx-Example']

In [6]:
# Load the sheet to inspect its content
gearbox_v4_df = excel_data.parse('GEARBOx v4')
# Display the first few rows of the dataframe to understand its structure
gearbox_v4_df.head()

,Parameter Name,Type,Description,Examples,MaxLength,Enum,ItemType,ItemDescription,ItemExamples,ItemEnum
0,age_min,integer,Does the eligibility criteria mention the mini...,2,10.0,NaN,NaN,NaN,NaN,NaN
1,age_max,integer,Does the eligibility criteria mention the maxi...,25,NaN,NaN,NaN,NaN,NaN,NaN
2,weight_min,number,Does the eligibility criteria mention the mini...,NaN,10.0,NaN,NaN,NaN,NaN,NaN
3,weight_max,number,Does the eligibility criteria mention the maxi...,NaN,10.0,NaN,NaN,NaN,NaN,NaN
4,sex,string,Does the eligibility criteria mention sexes of...,"""Male"", ""Female""",15.0,"""Male"", ""Female"", ""None""",NaN,NaN,NaN,NaN


In [7]:
# Function to generate class fields and enums for complex prompt
def generate_class_fields_and_enums(df, verb):
    fields = {}
    enum_classes = {}  # Dictionary to store generated enum classes as strings

    for _, row in df.iterrows():
        param_name = row['Parameter Name']

        # Determine the field type based on 'Type' and 'Enum' columns
        if row['Type'] == 'array':
            field_type = "Optional[List[str]]"  # Handle array type
        elif row['Type'] in ['integer', 'number']:
            field_type = "Optional[int]"
        elif pd.notna(row['Enum']) or pd.notna(row['ItemEnum']):
            # Handle Enum or ItemEnum
            field_type = f"Optional[List[{param_name.capitalize()}]]"
            # Choose between Enum or ItemEnum based on availability
            if pd.notna(row['Enum']):
                enum_items = [item.strip().strip('"') for item in row['Enum'].split(',')]
            elif pd.notna(row['ItemEnum']):
                enum_items = [item.strip().strip('"') for item in row['ItemEnum'].split(',')]

            # Create an enum class for this parameter
            enum_class_str = f"class {param_name.capitalize()}(Enum):\n"
            for item in enum_items:
                enum_name = re.sub(r"[\W\s]+", "_", item).upper()  # Ensure valid enum names
                enum_class_str += f"    {enum_name} = \"{item}\"\n"
            enum_classes[param_name.capitalize()] = enum_class_str
        else:
            field_type = "Optional[str]"

        # Combine description and examples if available
        description = row['Description']
        example = row['Examples']
        if pd.notna(example):
            description += f" Example: {example}"
        description = description.replace("{verb}", verb).replace('"', '\\"')

        # Add the field definition
        fields[param_name] = f"{field_type} = Field(\n        None, \n        description=\"{description}\")"

    return fields, enum_classes


# Generate class fields for complex prompt
class_fields_ic, enum_classes = generate_class_fields_and_enums(gearbox_v4_df, "included")
class_fields_ec, _ = generate_class_fields_and_enums(gearbox_v4_df, "excluded")

# Compile enum classes into the environment
# for enum_name, enum_definition in enum_classes.items():
#     exec(enum_definition)
class_enum = ""
for enum_name, enum_definition in enum_classes.items():
    class_enum += f"{enum_definition}\n"

exec(class_enum)

In [8]:
# Helper function to get dictionary chunks of size 2
def dict_chunk(dictionary, size=2):
    iterator = iter(dictionary.items())
    for chunk in iter(lambda: list(islice(iterator, size)), []):
        yield chunk

In [10]:
for item in abstract_list:
    file_name, inclusion_criteria, exclusion_criteria = item
    print(file_name)
    clean_text_ic = inclusion_criteria.replace("  ", " ").replace("\n", "; ").replace(';', ' ')
    clean_text_ec = exclusion_criteria.replace("  ", " ").replace("\n", "; ").replace(';', ' ')

    entity_value_list_ic = []
    for chunk in dict_chunk(class_fields_ic, size=2):
        class_definition = "class DynamicEntity(BaseModel):\n"
        for field_name, field_definition in chunk:
            class_definition += f"    {field_name}: {field_definition}\n"

        # Dynamically create and register the class
        exec(class_definition.strip(), globals())
        response_model_ic = globals()["DynamicEntity"]

        # Unified complex prompt
        complex_prompt_ic = f"""
        Extract the following information:
        {', '.join([field_name for field_name, _ in chunk])}.
        Criteria Text: {clean_text_ic}
        """
        try:
            # Query the AI directly with the complex prompt
            Classifier_IC = InfectionClassifier(response_model=response_model_ic, model_name=model, client=in_client)
            output_ic = Classifier_IC.output(criteria=complex_prompt_ic, type='Inclusion')
            entity_value_list_ic.append(output_ic)
        except Exception as e:
            print(f"Error processing fields {[field_name for field_name, _ in chunk]}: {e}")

    df_IC = pd.concat(entity_value_list_ic, ignore_index=True)

    entity_value_list_ec = []
    for chunk in dict_chunk(class_fields_ec, size=2):
        class_definition = "class DynamicEntity(BaseModel):\n"
        for field_name, field_definition in chunk:
            class_definition += f"    {field_name}: {field_definition}\n"

        # Dynamically create and register the class
        exec(class_definition.strip(), globals())
        response_model_ec = globals()["DynamicEntity"]

        # Unified complex prompt
        complex_prompt_ec = f"""
        Extract the following information:
        {', '.join([field_name for field_name, _ in chunk])}.
        Criteria Text: {clean_text_ec}
        """
        try:
            # Query the AI directly with the complex prompt
            Classifier_EC = InfectionClassifier(response_model=response_model_ec, model_name=model, client=in_client)
            output_ec = Classifier_EC.output(criteria=complex_prompt_ec, type='Exclusion')
            entity_value_list_ec.append(output_ec)
        except Exception as e:
            print(f"Error processing fields {[field_name for field_name, _ in chunk]}: {e}")

    df_EC = pd.concat(entity_value_list_ec, ignore_index=True)

    # Merge the two DataFrames on the 'Entity' column
    df_final = pd.merge(df_IC, df_EC, on='Entity', how='outer', suffixes=('_IC', '_EC'))
    df_final = df_final.replace([None, "None", pd.NA], "")
    output_excel_path = f'../results/{file_name}_output_data_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'
    df_final.to_excel(output_excel_path, index=False)

NCT04898894.txt
NCT04915612.txt
NCT05188170.txt
